# CFD-Drag-2D: U-Net with mirror augmentation (M10)

Trains the M8 U-Net with `--flip` (uy -> -uy under the mirror) for seeds 0-2 on the M7 field
dataset (output of the notebook `cfd-drag-2d-m7-fields`) and evaluates each run.
Two seeds run in parallel, one per T4. Output: `/kaggle/working/m10_unet/flip_seed<k>/`.

In [ ]:
REPO = "https://github.com/PradeepkrishnaGM/CFD-Drag-2D.git"
COMMIT = "main"   # branch or commit SHA
EPOCHS = 100
SEEDS = [0, 1, 2]
RUN = "/kaggle/working/m10_unet"
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf /tmp/repo && git clone -q $REPO /tmp/repo && git -C /tmp/repo checkout -q $COMMIT && git -C /tmp/repo log --oneline -1

In [ ]:
%cd /tmp/repo
!pip install -q -e . --no-deps && pip install -q pytest
import glob, os, subprocess, torch
n_gpu = torch.cuda.device_count()
print("torch", torch.__version__, "GPUs", n_gpu)
DATA = os.path.dirname(sorted(glob.glob("/kaggle/input/**/fields.npy", recursive=True))[0])
print("data:", DATA)
r = subprocess.run(["python", "-m", "pytest", "-q", "tests/test_field_surrogate.py", "tests/test_m10_training.py", "-k", "unet"],
                   capture_output=True, text=True)
print(r.stdout[-800:])
assert r.returncode == 0, "tests failed"

In [ ]:
os.makedirs(RUN, exist_ok=True)
pending = list(SEEDS)
while pending:
    batch, pending = pending[:max(n_gpu, 1)], pending[max(n_gpu, 1):]
    procs = []
    for gpu, seed in enumerate(batch):
        out = f"{RUN}/flip_seed{seed}"
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
        log = open(f"{RUN}/train_seed{seed}.log", "w")
        procs.append(subprocess.Popen(["python", "scripts/train_fields.py", "--data", DATA, "--out", out,
                                       "--epochs", str(EPOCHS), "--device", "cuda", "--seed", str(seed), "--flip"],
                                      env=env, stdout=log, stderr=subprocess.STDOUT))
    for p in procs:
        assert p.wait() == 0, "training failed"
    print("done:", batch)
!tail -2 $RUN/train_seed*.log

In [ ]:
for seed in SEEDS:
    out = f"{RUN}/flip_seed{seed}"
    !python scripts/evaluate_fields.py --data $DATA --run $out | tail -8
    !git rev-parse HEAD > $out/commit.txt